# Clean Retrain and Full Evaluation
**Thesis:** Fine-Tuned YOLO11 for Ship Detection in the Istanbul Bosphorus Strait\
**Author:** Recep Ertugrul Eksi - Uskudar University, Computer Engineering (MSc)

This notebook produces the final evaluation for the revised thesis in a single run:

1. **Pre-flight checks**: dataset composition and a hard assertion that the train/validation pools share zero images with the held-out test set
2. **Training**: YOLO11s fine-tuned on the clean training split with the original hyperparameters (AdamW, lr0 0.001, cosine schedule, 50 epochs, 1088 px, seed 0, deterministic)
3. **Evaluation**: all four vanilla baselines plus the newly trained model on the 60-image test set, protocol identical to the original study (vanilla 640 px, fine-tuned 1088 px, greedy IoU matching at 0.5, no class filtering); the superseded December run is evaluated alongside for reference
4. **Analyses and figures**: confidence sweep, IoU sensitivity, precision-recall curves, English-label comparison grid, showcase detections, dataset statistics

**How to run:** Runtime > Change runtime type > select a GPU (L4 if available, T4 works), then Run all. Expect 30-50 minutes depending on GPU. Outputs land in Drive under `Comparisons/revision_outputs_clean/`.

In [ ]:
# 1. Mount Drive, set paths, PRE-FLIGHT CHECKS
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive/Uskudar Uni Master/YOLO Model klasor"
DS_ROOT = os.path.join(BASE, "Custom Dataset", "bogaz_v_1.v3i.yolov8")
TEST_DATASET = os.path.join(BASE, "Test Dataset")
TEST_IMAGES_PATH = os.path.join(TEST_DATASET, "images")
TEST_LABELS_PATH = os.path.join(TEST_DATASET, "labels")
RUNS_DIR = os.path.join(BASE, "runs")
OUTPUT_DIR = os.path.join(BASE, "Comparisons", "revision_outputs_clean")
OLD_MODEL_PATH = os.path.join(RUNS_DIR, "bosphorus_yolo11s_10884", "weights", "best.pt")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def img_names(d):
    return set(f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.jpeg', '.png'))) if os.path.isdir(d) else set()

train_imgs = img_names(os.path.join(DS_ROOT, 'train', 'images'))
valid_imgs = img_names(os.path.join(DS_ROOT, 'valid', 'images'))
test_imgs  = img_names(TEST_IMAGES_PATH)
print(f"train: {len(train_imgs)}  valid: {len(valid_imgs)}  test: {len(test_imgs)}")

overlap = (train_imgs | valid_imgs) & test_imgs
assert len(overlap) == 0, f"OVERLAP DETECTED between training pool and test set: {sorted(overlap)[:5]}..."
print("PASS: zero overlap between the training pool (train+valid) and the held-out test set.")
assert len(test_imgs) == 60, f"expected 60 test images, found {len(test_imgs)}"
print("PASS: test set has exactly 60 images.")

In [ ]:
# 2. Install Ultralytics
!pip install -q -U ultralytics

import ultralytics, torch
from ultralytics import YOLO
print("Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - training will be very slow, use a GPU runtime")

In [ ]:
# 3. Helper functions (evaluation protocol, dual-format ground-truth parser)
import numpy as np
import time
import cv2

def parse_yolo_label(label_path, img_width, img_height):
    # box lines: class xc yc w h (5 fields); polygon lines: class x1 y1 ... xn yn (odd count > 5)
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                xc, yc = float(parts[1]) * img_width, float(parts[2]) * img_height
                w, h = float(parts[3]) * img_width, float(parts[4]) * img_height
                boxes.append([xc - w/2, yc - h/2, xc + w/2, yc + h/2])
            elif len(parts) > 5 and len(parts) % 2 == 1:
                xs = [float(v) * img_width for v in parts[1::2]]
                ys = [float(v) * img_height for v in parts[2::2]]
                boxes.append([min(xs), min(ys), max(xs), max(ys)])
    return boxes

def calculate_iou(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1]); a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    u = a1 + a2 - inter
    return inter / u if u > 0 else 0

def match_boxes(pred_boxes, gt_boxes, iou_threshold=0.5):
    if not pred_boxes:
        return 0, 0, len(gt_boxes), []
    if not gt_boxes:
        return 0, len(pred_boxes), 0, []
    gt_matched = [False] * len(gt_boxes)
    matched_ious = []; tp = 0
    for pred in pred_boxes:
        best_iou = 0; best_idx = -1
        for gi, gt in enumerate(gt_boxes):
            if gt_matched[gi]:
                continue
            iou = calculate_iou(pred, gt)
            if iou > best_iou:
                best_iou = iou; best_idx = gi
        if best_iou >= iou_threshold and best_idx >= 0:
            gt_matched[best_idx] = True
            matched_ious.append(best_iou); tp += 1
    return tp, len(pred_boxes) - tp, len(gt_boxes) - tp, matched_ious

print("Helpers ready.")

In [ ]:
# 4. Verify test ground truth against the published evidence, then load it
from PIL import Image
import urllib.request, csv, io

test_images = sorted(img_names(TEST_IMAGES_PATH))
gt_cache = {}
for n in test_images:
    with Image.open(os.path.join(TEST_IMAGES_PATH, n)) as im:
        w, h = im.size
    gt_cache[n] = parse_yolo_label(os.path.join(TEST_LABELS_PATH, os.path.splitext(n)[0] + '.txt'), w, h)
total_gt = sum(len(v) for v in gt_cache.values())
print("Ground-truth vessels in test set:", total_gt)

EV_URL = "https://raw.githubusercontent.com/Mizar-107/Uskudar-University-Thesis-Repository/main/evidence/comparison_metrics_60images.csv"
with urllib.request.urlopen(EV_URL) as r:
    ev = {row['image']: int(row['gt_count']) for row in csv.DictReader(io.TextIOWrapper(r, encoding='utf-8'))}
bad = [(n, len(gt_cache.get(n, [])), g) for n, g in ev.items() if len(gt_cache.get(n, [])) != g]
assert not bad, f"GT mismatch vs published evidence: {bad[:5]}"
print(f"PASS: ground-truth counts match the published evidence for all {len(ev)} images.")

In [ ]:
# 5. Write a data yaml with absolute paths, then TRAIN (original hyperparameters, clean split)
import yaml

with open(os.path.join(DS_ROOT, 'data.yaml')) as f:
    data_cfg = yaml.safe_load(f)
data_cfg['path'] = DS_ROOT
data_cfg['train'] = 'train/images'
data_cfg['val'] = 'valid/images'
data_cfg.pop('test', None)
clean_yaml = os.path.join(DS_ROOT, 'data_clean.yaml')
with open(clean_yaml, 'w') as f:
    yaml.safe_dump(data_cfg, f)
print("Wrote", clean_yaml, "| classes:", data_cfg.get('names'))

model = YOLO('yolo11s.pt')
results = model.train(
    data=clean_yaml,
    epochs=50, patience=10, batch=-1, imgsz=1088,
    rect=True, cache=True, workers=2, amp=True,
    optimizer='AdamW', lr0=0.001, lrf=0.01, cos_lr=True,
    close_mosaic=10, seed=0, deterministic=True,
    project=RUNS_DIR, name='bosphorus_yolo11s_clean',
    verbose=True, plots=True,
)
CLEAN_MODEL_PATH = os.path.join(RUNS_DIR, 'bosphorus_yolo11s_clean', 'weights', 'best.pt')
print("\nTraining complete. Weights:", CLEAN_MODEL_PATH)

In [ ]:
# 6. Training summary from results.csv
import pandas as pd
res_csv = os.path.join(RUNS_DIR, 'bosphorus_yolo11s_clean', 'results.csv')
tr = pd.read_csv(res_csv)
tr.columns = [c.strip() for c in tr.columns]
last = tr.iloc[-1]
cols = [c for c in tr.columns if 'metrics' in c or c in ('epoch', 'time')]
print(tr[cols].tail(3).to_string(index=False))
print()
print(f"Final epoch {int(last['epoch'])}: valid precision {100*last['metrics/precision(B)']:.2f}%, recall {100*last['metrics/recall(B)']:.2f}%, mAP50 {100*last['metrics/mAP50(B)']:.2f}%, mAP50-95 {100*last['metrics/mAP50-95(B)']:.2f}%")
if 'time' in tr.columns:
    print(f"Total training time: {last['time']/60:.1f} minutes")
tr.to_csv(os.path.join(OUTPUT_DIR, 'training_results_clean.csv'), index=False)

In [ ]:
# 7. Load models and run the two inference passes per model over the test set
MODEL_SPECS = {
    'YOLOv8n':    {'weights': 'yolov8n.pt', 'imgsz': 640},
    'YOLOv8s':    {'weights': 'yolov8s.pt', 'imgsz': 640},
    'YOLO11n':    {'weights': 'yolo11n.pt', 'imgsz': 640},
    'YOLO11s':    {'weights': 'yolo11s.pt', 'imgsz': 640},
    'Fine-tuned (clean)': {'weights': CLEAN_MODEL_PATH, 'imgsz': 1088},
    'Superseded (Dec run)': {'weights': OLD_MODEL_PATH, 'imgsz': 1088},
}
models = {}
for name, spec in MODEL_SPECS.items():
    models[name] = YOLO(spec['weights'])
    print("Loaded", name)

cache = {}
for name, spec in MODEL_SPECS.items():
    cache[name] = {}
    t0 = time.time()
    for n in test_images:
        pth = os.path.join(TEST_IMAGES_PATH, n)
        entry = {}
        for key, conf in (('low', 0.01), ('p25', 0.25)):
            rs = models[name].predict(source=pth, conf=conf, imgsz=spec['imgsz'], classes=None, verbose=False)
            entry[key] = ([b.xyxy[0].cpu().numpy().tolist() for r in rs for b in r.boxes],
                          [float(b.conf[0]) for r in rs for b in r.boxes])
        cache[name][n] = entry
    print(f"{name}: {time.time()-t0:.1f}s")

In [ ]:
# 8. Confidence sweep + THE DECISION VIEW (old vs clean at the protocol threshold)
def aggregate(model_name, source_key, conf_filter=None, iou_thr=0.5):
    TP = FP = FN = 0; all_m = []
    for n in test_images:
        boxes, confs = cache[model_name][n][source_key]
        if conf_filter is not None:
            boxes = [b for b, c in zip(boxes, confs) if c >= conf_filter]
        tp, fp, fn, mi = match_boxes(boxes, gt_cache[n], iou_thr)
        TP += tp; FP += fp; FN += fn; all_m.extend(mi)
    P = TP / (TP + FP) if TP + FP else 0.0
    R = TP / (TP + FN) if TP + FN else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    return {'precision_%': round(P*100, 2), 'recall_%': round(R*100, 2), 'f1_%': round(F1*100, 2),
            'mean_iou': round(float(np.mean(all_m)) if all_m else 0.0, 3),
            'tp': TP, 'fp': FP, 'fn': FN, 'detections': TP + FP}

rows = []
for name in MODEL_SPECS:
    for t in [0.10, 0.25, 0.40, 0.50]:
        r = aggregate(name, 'p25') if t == 0.25 else aggregate(name, 'low', conf_filter=t)
        rows.append({'model': name, 'conf': t, **r})
sweep = pd.DataFrame(rows)
sweep.to_csv(os.path.join(OUTPUT_DIR, 'conf_sweep_clean.csv'), index=False)
print(sweep.to_string(index=False))

print()
print("=" * 70)
print("DECISION VIEW - test set, conf 0.25, IoU 0.5:")
for name in ['Superseded (Dec run)', 'Fine-tuned (clean)']:
    r = aggregate(name, 'p25')
    print(f"  {name:24s} recall {r['recall_%']:6.2f}%  precision {r['precision_%']:6.2f}%  f1 {r['f1_%']:6.2f}%  mean IoU {r['mean_iou']:.3f}")
print("  (December run had the test images in its training set; the clean run did not)")
print("=" * 70)

In [ ]:
# 9. IoU matching-threshold sensitivity (clean model and baselines, conf 0.25)
rows = []
for name in MODEL_SPECS:
    for it in [0.30, 0.50, 0.70]:
        rows.append({'model': name, 'iou_threshold': it, **aggregate(name, 'p25', iou_thr=it)})
iou_df = pd.DataFrame(rows)
iou_df.to_csv(os.path.join(OUTPUT_DIR, 'iou_sensitivity_clean.csv'), index=False)
print(iou_df.to_string(index=False))

In [ ]:
# 10. Precision-recall curves: clean fine-tuned vs strongest vanilla
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pts = [round(0.05 * i, 2) for i in range(1, 19)]
fig, ax = plt.subplots(figsize=(7, 5.2), dpi=150)
curve_rows = []
for name, style in [('Fine-tuned (clean)', '-o'), ('YOLO11s', '-s')]:
    P, R = [], []
    for t in pts:
        r = aggregate(name, 'low', conf_filter=t)
        P.append(r['precision_%']); R.append(r['recall_%'])
        curve_rows.append({'model': name, 'conf': t, **{k: r[k] for k in ('precision_%', 'recall_%', 'f1_%')}})
    ax.plot(R, P, style, markersize=3.5, linewidth=1.4, label=name)
ax.set_xlabel('Recall (%)'); ax.set_ylabel('Precision (%)')
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.grid(alpha=0.3); ax.legend()
ax.set_title('Precision-recall behaviour across confidence thresholds (60-image test set)')
fig.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, 'fig_pr_curves_clean.png'), bbox_inches='tight')
pd.DataFrame(curve_rows).to_csv(os.path.join(OUTPUT_DIR, 'pr_curve_points_clean.csv'), index=False)
plt.show(); print("saved")

In [ ]:
# 11. English-label comparison grid: four vanilla baselines + clean fine-tuned
import random
random.seed(42)
selected = random.sample(test_images, 8)
GRID_MODELS = ['YOLOv8n', 'YOLOv8s', 'YOLO11n', 'YOLO11s', 'Fine-tuned (clean)']
CELL_W = 520

def draw_panel(img_path, boxes, confs, gts, is_ft):
    img = cv2.imread(img_path)
    gm = [False] * len(gts); flags = []
    for b in boxes:
        bi = 0; bx = -1
        for gi, g in enumerate(gts):
            if gm[gi]:
                continue
            iou = calculate_iou(b, g)
            if iou > bi:
                bi = iou; bx = gi
        ok = bi >= 0.5 and bx >= 0
        if ok:
            gm[bx] = True
        flags.append(ok)
    for b, c, ok in zip(boxes, confs, flags):
        col = (60, 200, 60) if ok else (50, 50, 230)
        x1, y1, x2, y2 = [int(v) for v in b]
        cv2.rectangle(img, (x1, y1), (x2, y2), col, max(2, img.shape[1] // 640))
        lab = f"ship {c:.2f}" if is_ft else f"{c:.2f}"
        fs = max(0.5, img.shape[1] / 1600)
        (tw, th), _ = cv2.getTextSize(lab, cv2.FONT_HERSHEY_SIMPLEX, fs, 2)
        ty = y1 - 6 if y1 - th - 8 > 0 else y2 + th + 6
        cv2.rectangle(img, (x1, ty - th - 4), (x1 + tw + 4, ty + 4), col, -1)
        cv2.putText(img, lab, (x1 + 2, ty), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 255), 2)
    h, w = img.shape[:2]
    return cv2.resize(img, (CELL_W, int(h * CELL_W / w)))

grid_rows = []
for n in selected:
    pth = os.path.join(TEST_IMAGES_PATH, n)
    panels = []
    for m in GRID_MODELS:
        boxes, confs = cache[m][n]['p25']
        panels.append(draw_panel(pth, boxes, confs, gt_cache[n], m == 'Fine-tuned (clean)'))
    hmax = max(p.shape[0] for p in panels)
    panels = [cv2.copyMakeBorder(p, 0, hmax - p.shape[0], 0, 4, cv2.BORDER_CONSTANT, value=(255,255,255)) for p in panels]
    grid_rows.append(np.hstack(panels))
header = np.full((46, grid_rows[0].shape[1], 3), 255, dtype=np.uint8)
for i, m in enumerate(GRID_MODELS):
    cv2.putText(header, m, (i * (CELL_W + 4) + 14, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (30, 30, 30), 2)
grid = np.vstack([header] + grid_rows)
cv2.imwrite(os.path.join(OUTPUT_DIR, 'comparison_grid_clean_en.png'), grid)
# print-resolution jpeg for document embedding
cv2.imwrite(os.path.join(OUTPUT_DIR, 'comparison_grid_clean_en_print.jpg'), grid, [cv2.IMWRITE_JPEG_QUALITY, 90])
print("Saved grid", grid.shape, "- green = correct (IoU >= 0.5), red = false positive, labels in English")

In [ ]:
# 12. Showcase figures: clean model on the three densest scenes
dense = sorted(test_images, key=lambda n: len(gt_cache[n]), reverse=True)[:3]
for i, n in enumerate(dense, 1):
    boxes, confs = cache['Fine-tuned (clean)'][n]['p25']
    panel = draw_panel(os.path.join(TEST_IMAGES_PATH, n), boxes, confs, gt_cache[n], True)
    cv2.imwrite(os.path.join(OUTPUT_DIR, f'fig_showcase_clean_{i}.png'), panel)
    print(f"showcase {i}: {n} ({len(gt_cache[n])} vessels, {len(boxes)} detections)")

In [ ]:
# 13. Dataset statistics (dual-format parser)
import glob as _glob, json as _json

stats = {}; all_counts, all_areas, all_ars, centers = [], [], [], []
for split in ['train', 'valid']:
    lbl_dir = os.path.join(DS_ROOT, split, 'labels')
    img_dir = os.path.join(DS_ROOT, split, 'images')
    lf_list = sorted(_glob.glob(os.path.join(lbl_dir, '*.txt')))
    n_boxes = 0
    for lf in lf_list:
        cnt = 0
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cnt += 1; wn, hn = float(parts[3]), float(parts[4]); cx, cy = float(parts[1]), float(parts[2])
                elif len(parts) > 5 and len(parts) % 2 == 1:
                    cnt += 1
                    xs = [float(v) for v in parts[1::2]]; ys = [float(v) for v in parts[2::2]]
                    wn, hn = max(xs) - min(xs), max(ys) - min(ys)
                    cx, cy = (max(xs) + min(xs)) / 2, (max(ys) + min(ys)) / 2
                else:
                    continue
                all_areas.append(wn * hn); all_ars.append(wn / hn if hn > 0 else 0); centers.append((cx, cy))
        n_boxes += cnt; all_counts.append(cnt)
    stats[split] = {'images': len(img_names(img_dir)), 'boxes': n_boxes}
    print(split, stats[split])
test_boxes = sum(len(v) for v in gt_cache.values())
stats['test'] = {'images': len(test_images), 'boxes': test_boxes}
print('test', stats['test'])

fig, axes = plt.subplots(2, 2, figsize=(11, 8), dpi=140)
axes[0,0].hist(all_counts, bins=range(0, max(all_counts)+2), color='#2b6f8c', edgecolor='white')
axes[0,0].set_title('Annotated vessels per image (training pool)'); axes[0,0].set_xlabel('boxes'); axes[0,0].set_ylabel('images')
axes[0,1].hist([a*100 for a in all_areas], bins=40, color='#2b6f8c', edgecolor='white', log=True)
axes[0,1].set_title('Bounding-box area (% of image, log scale)'); axes[0,1].set_xlabel('% of image area')
axes[1,0].hist(all_ars, bins=40, range=(0, 8), color='#2b6f8c', edgecolor='white')
axes[1,0].set_title('Box aspect ratio (width/height)'); axes[1,0].set_xlabel('aspect ratio')
hb = axes[1,1].hist2d([c[0] for c in centers], [c[1] for c in centers], bins=40, cmap='viridis')
axes[1,1].set_title('Box-centre spatial density'); axes[1,1].invert_yaxis(); fig.colorbar(hb[3], ax=axes[1,1])
fig.suptitle('Dataset statistics: bogaz_v_1 training pool', y=1.0)
fig.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, 'fig_dataset_stats_clean.png'), bbox_inches='tight')
plt.show()

summary = {'splits': stats,
           'training_pool_images': stats['train']['images'] + stats['valid']['images'],
           'training_pool_boxes': stats['train']['boxes'] + stats['valid']['boxes'],
           'total_images_incl_test': stats['train']['images'] + stats['valid']['images'] + stats['test']['images'],
           'total_boxes_incl_test': stats['train']['boxes'] + stats['valid']['boxes'] + stats['test']['boxes'],
           'boxes_per_image_mean': float(np.mean(all_counts)), 'boxes_per_image_median': float(np.median(all_counts)),
           'boxes_per_image_max': int(max(all_counts)),
           'box_area_pct_median': float(np.median(all_areas))*100, 'box_area_pct_mean': float(np.mean(all_areas))*100,
           'aspect_ratio_median': float(np.median(all_ars))}
with open(os.path.join(OUTPUT_DIR, 'dataset_stats_clean.json'), 'w') as f:
    _json.dump(summary, f, indent=2)
print(_json.dumps(summary, indent=2))

In [ ]:
# 14. Package and finish
import shutil
shutil.make_archive(os.path.join(OUTPUT_DIR, '..', 'revision_outputs_clean'), 'zip', OUTPUT_DIR)
print("Files in", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    print("  -", f, f"({os.path.getsize(os.path.join(OUTPUT_DIR, f))//1024} KB)")
print()
print("Done. Clean weights:", CLEAN_MODEL_PATH)
print("All outputs in Drive under Comparisons/revision_outputs_clean/.")